# 🚀 Advanced RAG Experiments with LlamaIndex

This notebook walks through **three Advanced RAG techniques** as separate experiments:

| # | Technique | Stage | Description |
|---|-----------|-------|-------------|
| 1 | **HyDE** | Pre-retrieval | Generates a hypothetical answer → embeds that for retrieval |
| 2 | **Re-ranking** | Post-retrieval | Cross-encoder rescores retrieved chunks |
| 3 | **Sub-Question** | Pre-retrieval | Decomposes complex queries into sub-questions |

We compare each against a **Baseline** (naive vector retrieval).

---
## 1. Setup & Configuration

In [2]:
import os
import nest_asyncio

nest_asyncio.apply()

from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
# Set your Gemini API key
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY_HERE"  # Replace with your key

In [ ]:
# Configure global settings
Settings.llm = GoogleGenAI(model="gemini-2.5-flash-preview-04-17")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print(f"LLM: {Settings.llm.model}")
print(f"Embed model: {Settings.embed_model.model_name}")

---
## 2. Load Documents & Build Index

In [ ]:
# Load documents from the data/ folder
documents = SimpleDirectoryReader("../data").load_data()
print(f"Loaded {len(documents)} document(s)")

# Build the vector index
index = VectorStoreIndex.from_documents(documents=documents)
print("Index built successfully!")

---
## 3. Baseline — Naive Vector Retrieval

Simple vector similarity search with no enhancements. This serves as our control experiment.

In [ ]:
# Create a baseline query engine
baseline_engine = index.as_query_engine(similarity_top_k=5)

# Test query
TEST_QUERY = "What are the main topics discussed in the documents?"

baseline_response = baseline_engine.query(TEST_QUERY)
print("=" * 80)
print("BASELINE RESPONSE")
print("=" * 80)
print(baseline_response.response)
print(f"\nSource nodes: {len(baseline_response.source_nodes)}")
for i, node in enumerate(baseline_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] score={score:.4f} | {node.text[:100]}...")

---
## 4. Experiment 1: HyDE (Hypothetical Document Embeddings)

### How it works
1. The LLM generates a **hypothetical answer** to the query (even though it doesn't have access to the documents yet).
2. That hypothetical answer is **embedded** and used for vector search instead of the raw query.
3. Since the hypothetical answer is more semantically similar to actual document passages than a short query, retrieval quality improves.

### When to use
- Queries that are short or abstract (e.g., "What is RAG?")
- When there's a vocabulary mismatch between queries and documents

In [ ]:
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.core.indices.query.query_transform import HyDEQueryTransform

In [ ]:
# Create the HyDE transform
hyde_transform = HyDEQueryTransform(include_original=True)

# Let's see what the hypothetical document looks like
hyde_query_bundle = hyde_transform(TEST_QUERY)

print("=" * 80)
print("HyDE — HYPOTHETICAL DOCUMENT GENERATED")
print("=" * 80)
print(f"Original query: {TEST_QUERY}")
print(f"\nHypothetical document(s):")
for i, emb_str in enumerate(hyde_query_bundle.embedding_strs):
    print(f"\n--- Hypothetical Doc {i+1} ---")
    print(emb_str[:500])

In [ ]:
# Wrap the baseline engine with HyDE
hyde_engine = TransformQueryEngine(baseline_engine, hyde_transform)

hyde_response = hyde_engine.query(TEST_QUERY)
print("=" * 80)
print("HyDE RESPONSE")
print("=" * 80)
print(hyde_response.response)
print(f"\nSource nodes: {len(hyde_response.source_nodes)}")
for i, node in enumerate(hyde_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] score={score:.4f} | {node.text[:100]}...")

---
## 5. Experiment 2: Re-ranking (Cross-Encoder)

### How it works
1. Vector search retrieves **top-K** candidate chunks (fast but approximate).
2. A **cross-encoder** model scores each `(query, chunk)` pair jointly — this is much more accurate than bi-encoder similarity.
3. Chunks are re-sorted by the cross-encoder score, and the **top-N** are kept.

### Model
We use `BAAI/bge-reranker-v2-m3` — a free, local cross-encoder. No API key needed.

### When to use
- When initial retrieval returns partially relevant results
- When precision matters more than recall

In [ ]:
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank

In [ ]:
# Create the re-ranker (downloads model on first run)
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-v2-m3",
    top_n=3,  # Keep top 3 after re-ranking
)

# Create engine with re-ranking as a node postprocessor
rerank_engine = index.as_query_engine(
    similarity_top_k=5,  # Retrieve 5 candidates
    node_postprocessors=[reranker],  # Re-rank to keep top 3
)

rerank_response = rerank_engine.query(TEST_QUERY)
print("=" * 80)
print("RE-RANKING RESPONSE")
print("=" * 80)
print(rerank_response.response)
print(f"\nSource nodes (after re-ranking): {len(rerank_response.source_nodes)}")
for i, node in enumerate(rerank_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] score={score:.4f} | {node.text[:100]}...")

In [ ]:
# Compare scores: before vs after re-ranking
print("=" * 80)
print("SCORE COMPARISON: Baseline vs Re-ranked")
print("=" * 80)

print("\nBaseline (vector similarity scores):")
for i, node in enumerate(baseline_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] {score:.4f} | {node.text[:80]}...")

print("\nRe-ranked (cross-encoder scores):")
for i, node in enumerate(rerank_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    print(f"  [{i}] {score:.4f} | {node.text[:80]}...")

---
## 6. Experiment 3: Sub-Question Query Engine

### How it works
1. The LLM analyzes the user's question and **decomposes** it into independent sub-questions.
2. Each sub-question is sent to the appropriate query engine tool.
3. All sub-answers are **synthesized** into a single coherent response.

### When to use
- Complex, multi-part questions (e.g., "Compare X and Y", "What are the pros and cons of Z?")
- When a single retrieval pass wouldn't cover all aspects

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata

In [ ]:
# Wrap the baseline engine as a tool
query_engine_tools = [
    QueryEngineTool(
        query_engine=baseline_engine,
        metadata=ToolMetadata(
            name="document_search",
            description=(
                "Provides information from the loaded documents. "
                "Use this tool to answer questions about the document content."
            ),
        ),
    ),
]

# Create the sub-question engine
sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    use_async=True,
)

# Use a complex, multi-part query to show decomposition
COMPLEX_QUERY = "What are the main topics discussed and how do they relate to each other?"

sub_q_response = sub_question_engine.query(COMPLEX_QUERY)
print("=" * 80)
print("SUB-QUESTION ENGINE RESPONSE")
print("=" * 80)
print(sub_q_response.response)
print(f"\nSource nodes: {len(sub_q_response.source_nodes)}")
for i, node in enumerate(sub_q_response.source_nodes, 1):
    score = getattr(node, 'score', None)
    score_str = f"score={score:.4f}" if score is not None else "score=N/A"
    print(f"  [{i}] {score_str} | {node.text[:100]}...")

---
## 7. Comparison: All Techniques Side-by-Side

Let's compare all four approaches on the same query.

In [ ]:
COMPARISON_QUERY = "What are the key concepts explained in these documents?"

results = {}

print("Running Baseline...")
results["Baseline"] = baseline_engine.query(COMPARISON_QUERY)

print("Running HyDE...")
results["HyDE"] = hyde_engine.query(COMPARISON_QUERY)

print("Running Re-ranking...")
results["Re-ranking"] = rerank_engine.query(COMPARISON_QUERY)

print("Running Sub-Question...")
results["Sub-Question"] = sub_question_engine.query(COMPARISON_QUERY)

print("Done!")

In [ ]:
# Print side-by-side comparison
for technique, response in results.items():
    print("=" * 80)
    print(f"  {technique.upper()}")
    print("=" * 80)
    print(response.response)
    num_nodes = len(response.source_nodes) if hasattr(response, 'source_nodes') else 0
    print(f"\n  [Source nodes: {num_nodes}]")
    print()

---
## Summary

| Technique | Best For | Trade-off |
|-----------|----------|----------|
| **Baseline** | Simple, well-formed queries | Fast but limited |
| **HyDE** | Short/abstract queries, vocabulary mismatch | Extra LLM call per query |
| **Re-ranking** | Improving precision from noisy retrieval | Extra model inference (cross-encoder) |
| **Sub-Question** | Complex multi-part questions | Multiple LLM calls, slower |